In [7]:
# =========================================================
# Parameters and imports
# =========================================================
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.storagelevel import StorageLevel
import time

# ============================================
# PARAMETERS
# ============================================
run_id = "run_q2_001"
simulate_large_data = True
multiplier = 10000000   # increase carefully based on Fabric capacity
salt_buckets = 50   # use 4 first in trial capacity

print(f"run_id = {run_id}")
print(f"simulate_large_data = {simulate_large_data}")
print(f"multiplier = {multiplier}")
print(f"salt_buckets = {salt_buckets}")

StatementMeta(, 4ae7088e-3de5-42cf-b853-452883a403c7, 67, Finished, Available, Finished, False)

run_id = run_q2_001
simulate_large_data = True
multiplier = 10000000
salt_buckets = 50


In [9]:
# =========================================================
# Read source tables
# =========================================================
fact_sales = spark.read.table("dbo.fact_sales")
fact_returns = spark.read.table("dbo.fact_returns")
dim_product = spark.read.table("dbo.dim_product")
dim_customer = spark.read.table("dbo.dim_customer")
dim_date = spark.read.table("dbo.dim_date")

print("Base tables loaded")

StatementMeta(, 4ae7088e-3de5-42cf-b853-452883a403c7, 69, Finished, Available, Finished, False)

Base tables loaded


In [10]:
# =========================================================
# Simulate larger dataset if needed
# =========================================================
if simulate_large_data:
    numbers_df = spark.range(multiplier).withColumnRenamed("id", "multiplier_id")

    fact_sales_large = fact_sales.crossJoin(numbers_df)
    fact_returns_large = fact_returns.crossJoin(numbers_df)

    print("Simulated larger datasets created")
else:
    fact_sales_large = fact_sales
    fact_returns_large = fact_returns
    print("Using original datasets")

StatementMeta(, 4ae7088e-3de5-42cf-b853-452883a403c7, 70, Finished, Available, Finished, False)

Simulated larger datasets created


In [11]:
# =========================================================
# Optional skew check
# =========================================================
print("Top skewed product_id values in fact_sales:")
fact_sales_large.groupBy("product_id") \
    .count() \
    .orderBy(F.col("count").desc()) \
    .show(10, truncate=False)

StatementMeta(, 4ae7088e-3de5-42cf-b853-452883a403c7, 71, Finished, Available, Finished, False)

Top skewed product_id values in fact_sales:
+----------+--------+
|product_id|count   |
+----------+--------+
|101       |20000000|
|104       |10000000|
|102       |10000000|
|103       |10000000|
+----------+--------+



In [12]:
# =========================================================
# Naive approach
# =========================================================
start_naive = time.time()

# --------------------------------------------
# Daily Sales by Region
# --------------------------------------------
naive_daily_sales_by_region = (
    fact_sales_large
    .groupBy("order_date", "region")
    .agg(F.sum("line_amount").alias("daily_sales_amount"))
)

# --------------------------------------------
# Top 10 Products by Category
# --------------------------------------------
naive_top_products_by_category = (
    fact_sales_large.alias("f")
    .join(dim_product.alias("p"), "product_id", "left")
    .groupBy("category", "product_id", "product_name")
    .agg(F.sum("line_amount").alias("sales_amount"))
)

top_window = Window.partitionBy("category").orderBy(F.col("sales_amount").desc())

naive_top_products_by_category = (
    naive_top_products_by_category
    .withColumn("rn", F.row_number().over(top_window))
    .filter(F.col("rn") <= 10)
    .drop("rn")
)

# --------------------------------------------
# Customer Lifetime Value
# --------------------------------------------
naive_customer_lifetime_value = (
    fact_sales_large
    .groupBy("customer_id")
    .agg(
        F.sum("line_amount").alias("customer_lifetime_value"),
        F.countDistinct("order_id").alias("total_orders")
    )
)

# --------------------------------------------
# Monthly Return Rate
# --------------------------------------------
naive_monthly_sales = (
    fact_sales_large
    .withColumn("sales_month", F.date_format("order_date", "yyyy-MM"))
    .groupBy("sales_month")
    .agg(F.sum("line_amount").alias("sales_amount"))
)

naive_monthly_returns = (
    fact_returns_large
    .withColumn("return_month", F.date_format("return_date", "yyyy-MM"))
    .groupBy("return_month")
    .agg(F.sum("return_qty").alias("return_qty"))
)

naive_monthly_return_rate = (
    naive_monthly_sales.alias("s")
    .join(
        naive_monthly_returns.alias("r"),
        F.col("s.sales_month") == F.col("r.return_month"),
        "left"
    )
    .select(
        F.col("s.sales_month").alias("month"),
        F.col("sales_amount"),
        F.coalesce(F.col("return_qty"), F.lit(0)).alias("return_qty"),
        F.when(
            F.col("sales_amount") != 0,
            F.col("return_qty") / F.col("sales_amount")
        ).otherwise(F.lit(0)).alias("monthly_return_rate")
    )
)

# --------------------------------------------
# Write outputs
# --------------------------------------------
naive_daily_sales_by_region.write.mode("overwrite").format("delta").saveAsTable("dbo.q2_naive_daily_sales_by_region")
naive_top_products_by_category.write.mode("overwrite").format("delta").saveAsTable("dbo.q2_naive_top_10_products_by_category")
naive_customer_lifetime_value.write.mode("overwrite").format("delta").saveAsTable("dbo.q2_naive_customer_lifetime_value")
naive_monthly_return_rate.write.mode("overwrite").format("delta").saveAsTable("dbo.q2_naive_monthly_return_rate")

end_naive = time.time()
naive_runtime = end_naive - start_naive

print(f"Naive runtime: {naive_runtime:.2f} seconds")

StatementMeta(, 4ae7088e-3de5-42cf-b853-452883a403c7, 72, Finished, Available, Finished, False)

Naive runtime: 32.60 seconds


In [13]:
# =========================================================
# Optimized approach with salting
# =========================================================

start_opt = time.time()

# --------------------------------------------
# 1. Predicate pushdown / early filtering
# --------------------------------------------
fact_sales_opt = (
    fact_sales_large
    .filter(F.col("order_date").isNotNull())
    .filter(F.col("line_amount").isNotNull())
    .filter(F.col("product_id").isNotNull())
    .filter(F.col("customer_id").isNotNull())
)

fact_returns_opt = (
    fact_returns_large
    .filter(F.col("return_date").isNotNull())
    .filter(F.col("return_qty").isNotNull())
    .filter(F.col("product_id").isNotNull())
)

# --------------------------------------------
# 2. Persist reused dataframes
# --------------------------------------------
fact_sales_opt = fact_sales_opt.persist(StorageLevel.MEMORY_AND_DISK)
fact_returns_opt = fact_returns_opt.persist(StorageLevel.MEMORY_AND_DISK)

# --------------------------------------------
# 3. Broadcast small dimensions
# --------------------------------------------
dim_product_b = F.broadcast(dim_product.select("product_id", "product_name", "category"))
dim_customer_b = F.broadcast(dim_customer.select("customer_id"))

# --------------------------------------------
# 4. Daily Sales by Region
# --------------------------------------------
optimized_daily_sales_by_region = (
    fact_sales_opt
    .groupBy("order_date", "region")
    .agg(F.sum("line_amount").alias("daily_sales_amount"))
)

# --------------------------------------------
# 5. Top 10 Products by Category with salting
# --------------------------------------------
fact_sales_salted = fact_sales_opt.withColumn(
    "salt",
    (F.rand() * salt_buckets).cast("int")
)

salt_df = spark.range(0, salt_buckets).toDF("salt")

dim_product_salted = (
    dim_product.select("product_id", "product_name", "category")
    .crossJoin(salt_df)
)

optimized_top_products_by_category = (
    fact_sales_salted.alias("f")
    .join(
        F.broadcast(dim_product_salted).alias("p"),
        (F.col("f.product_id") == F.col("p.product_id")) &
        (F.col("f.salt") == F.col("p.salt")),
        "left"
    )
    .groupBy(
        F.col("p.category").alias("category"),
        F.col("f.product_id").alias("product_id"),
        F.col("p.product_name").alias("product_name")
    )
    .agg(F.sum("line_amount").alias("sales_amount"))
)

top_window = Window.partitionBy("category").orderBy(F.col("sales_amount").desc())

optimized_top_products_by_category = (
    optimized_top_products_by_category
    .withColumn("rn", F.row_number().over(top_window))
    .filter(F.col("rn") <= 10)
    .drop("rn")
)

# --------------------------------------------
# 6. Customer Lifetime Value
# --------------------------------------------
optimized_customer_lifetime_value = (
    fact_sales_opt
    .groupBy("customer_id")
    .agg(
        F.sum("line_amount").alias("customer_lifetime_value"),
        F.countDistinct("order_id").alias("total_orders")
    )
)

# --------------------------------------------
# 7. Monthly Return Rate
# --------------------------------------------
optimized_monthly_sales = (
    fact_sales_opt
    .withColumn("sales_month", F.date_format("order_date", "yyyy-MM"))
    .groupBy("sales_month")
    .agg(F.sum("line_amount").alias("sales_amount"))
)

optimized_monthly_returns = (
    fact_returns_opt
    .withColumn("return_month", F.date_format("return_date", "yyyy-MM"))
    .groupBy("return_month")
    .agg(F.sum("return_qty").alias("return_qty"))
)

optimized_monthly_return_rate = (
    optimized_monthly_sales.alias("s")
    .join(
        optimized_monthly_returns.alias("r"),
        F.col("s.sales_month") == F.col("r.return_month"),
        "left"
    )
    .select(
        F.col("s.sales_month").alias("month"),
        F.col("sales_amount"),
        F.coalesce(F.col("return_qty"), F.lit(0)).alias("return_qty"),
        F.when(
            F.col("sales_amount") != 0,
            F.col("return_qty") / F.col("sales_amount")
        ).otherwise(F.lit(0)).alias("monthly_return_rate")
    )
)

# --------------------------------------------
# 8. Repartition before write
# --------------------------------------------
optimized_daily_sales_by_region.repartition("region") \
    .write.mode("overwrite").format("delta").saveAsTable("dbo.q2_opt_daily_sales_by_region")

optimized_top_products_by_category.repartition("category") \
    .write.mode("overwrite").format("delta").saveAsTable("dbo.q2_opt_top_10_products_by_category")

optimized_customer_lifetime_value.repartition(8) \
    .write.mode("overwrite").format("delta").saveAsTable("dbo.q2_opt_customer_lifetime_value")

optimized_monthly_return_rate.repartition(4) \
    .write.mode("overwrite").format("delta").saveAsTable("dbo.q2_opt_monthly_return_rate")

end_opt = time.time()
optimized_runtime = end_opt - start_opt

print(f"Optimized runtime: {optimized_runtime:.2f} seconds")

StatementMeta(, 4ae7088e-3de5-42cf-b853-452883a403c7, 73, Finished, Available, Finished, False)

Optimized runtime: 62.47 seconds


In [14]:
# =========================================================
# Runtime comparison table
# =========================================================
comparison_data = [
    ("naive", naive_runtime),
    ("optimized", optimized_runtime)
]

comparison_df = spark.createDataFrame(comparison_data, ["approach", "runtime_seconds"])
comparison_df.show()

comparison_df.write.mode("overwrite").format("delta").saveAsTable("dbo.q2_runtime_comparison")

StatementMeta(, 4ae7088e-3de5-42cf-b853-452883a403c7, 74, Finished, Available, Finished, False)

+---------+-----------------+
| approach|  runtime_seconds|
+---------+-----------------+
|    naive|32.60159635543823|
|optimized|62.46747398376465|
+---------+-----------------+



In [15]:
# =========================================================
# Optional compaction / optimize
# =========================================================
try:
    spark.sql("OPTIMIZE dbo.q2_opt_daily_sales_by_region")
    spark.sql("OPTIMIZE dbo.q2_opt_top_10_products_by_category")
    spark.sql("OPTIMIZE dbo.q2_opt_customer_lifetime_value")
    spark.sql("OPTIMIZE dbo.q2_opt_monthly_return_rate")
    print("OPTIMIZE completed")
except Exception as e:
    print("OPTIMIZE not supported or skipped:", str(e))

StatementMeta(, 4ae7088e-3de5-42cf-b853-452883a403c7, 75, Finished, Available, Finished, False)

OPTIMIZE completed
